In [1]:
import random
import ipywidgets as widgets
from IPython.display import display, clear_output
from qiskit import QuantumCircuit, execute, Aer
from qiskit.providers.aer import QasmSimulator
import numpy as np

global ciphertext_bits, encryption_key
ciphertext_bits = ""
encryption_key = ""

def alice_state_prep(state, basis):
    num_qubits = len(state)
    circuit = QuantumCircuit(num_qubits)
    for i in range(len(basis)):
        if state[i] == 1:
            circuit.x(i)
        if basis[i] == 1:
            circuit.h(i)
    return circuit

def bob_measurement(circuit, basis):
    for i in range(len(basis)):
        if basis[i] == 1:
            circuit.h(i)
    circuit.measure_all()

def key_creation(circuit, alice_basis, bob_basis):
    backend = QasmSimulator()
    result = execute(circuit, backend=backend, shots=1).result()
    counts = result.get_counts()
    key = list(counts.keys())[0] if counts else ""
    encryption_key = ''.join([key[i] for i in range(min(len(alice_basis), len(key))) if alice_basis[i] == bob_basis[i]])
    return encryption_key

def text_to_bits(text):
    return ''.join(format(ord(char), '08b') for char in text)

def encrypt_message(_):
    global ciphertext_bits, encryption_key
    plaintext = text_input.value.strip()
    if not plaintext:
        with output:
            clear_output(wait=True)
            print("Please enter a message to encrypt.")
        return
    
    plain_text_bits = text_to_bits(plaintext)
    #num_qubits = max(len(plain_text_bits), 32)  # Ensure at least 32 qubits
    num_qubits=int(2.0*len(plain_text_bits))
    print(num_qubits)
    alice_basis = np.random.randint(2, size=num_qubits)
    alice_state = np.random.randint(2, size=num_qubits)
    cir = alice_state_prep(alice_state, alice_basis)
    bob_basis = np.random.randint(2, size=num_qubits)
    bob_measurement(cir, bob_basis)
    encryption_key = key_creation(cir, alice_basis, bob_basis)
    encryption_key = encryption_key[:len(plain_text_bits)]
    ciphertext_bits = ''.join(str(int(plain_text_bits[i]) ^ int(encryption_key[i])) for i in range(len(plain_text_bits)))
    
    clear_output(wait=True)
    display(text_input, encrypt_button, decrypt_button, output)
    with output:
        print("Ciphertext (in binary):", ciphertext_bits)
        print("Key:", encryption_key)

def bits_to_text(bits):
    chars = [chr(int(bits[i:i+8], 2)) for i in range(0, len(bits), 8)]
    return ''.join(chars)

def decrypt_message(_):
    global ciphertext_bits, encryption_key
    if not ciphertext_bits or not encryption_key:
        with output:
            clear_output(wait=True)
            print("No encrypted message found. Please encrypt a message first.")
        return
    decrypted_bits = ''.join(str(int(ciphertext_bits[i]) ^ int(encryption_key[i])) for i in range(len(ciphertext_bits)))
    decrypted_text = bits_to_text(decrypted_bits)
    clear_output(wait=True)
    display(text_input, encrypt_button, decrypt_button, output)
    with output:
        print("Decrypted Text:", decrypted_text)

text_input = widgets.Text(description="Enter Text:", placeholder="Type your message here")

encrypt_button = widgets.Button(description="Encrypt Message")

decrypt_button = widgets.Button(description="Decrypt Message")

encrypt_button.on_click(encrypt_message)

decrypt_button.on_click(decrypt_message)

output = widgets.Output()

display(text_input, encrypt_button, decrypt_button, output)
#display(text_input, encrypt_button, output)


Text(value='', description='Enter Text:', placeholder='Type your message here')

Button(description='Encrypt Message', style=ButtonStyle())

Button(description='Decrypt Message', style=ButtonStyle())

Output()